In [ ]:
# SHAM_SECRETS_FALLBACK_V3 (2026-10-15)
# بعد ساعات من التدريب فشلت خلية الحفظ بـ NameError: '_sf_orig' / '_sf_creds'
# (الغلاف القديم V1 يعتمد على متغيرات عامة اختفت). هذه الخلية مستقلة تماماً:
# لا تعتمد على أي متغير عام، تضبط KAGGLE_USERNAME/KAGGLE_KEY في البيئة و
# ~/.kaggle/kaggle.json فوراً، وتلتقط أي خطأ من الأغلفة السابقة (حتى NameError)
# فترجع القيم من البيئة. آمنة إن أُعيد تشغيلها.
def _sham_secrets_fallback_v3():
    import glob, json, os

    def load_creds(_glob=glob, _json=json, _os=os):
        u, k = _os.environ.get("KAGGLE_USERNAME"), _os.environ.get("KAGGLE_KEY") or _os.environ.get("KAGGLE_API_TOKEN")
        if u and k:
            return {"username": u, "key": k}
        pats = ["/kaggle/input/sham-runtime-creds/**/kaggle.json", "/kaggle/input/**/kaggle.json",
                _os.path.expanduser("~/.kaggle/kaggle.json")]
        for pat in pats:
            for p in sorted(_glob.glob(pat, recursive=True)):
                try:
                    with open(p) as f:
                        d = _json.load(f)
                    if d.get("username") and d.get("key"):
                        return d
                except Exception:
                    pass
        return {}

    c = load_creds()
    if c:
        os.environ.setdefault("KAGGLE_USERNAME", c["username"])
        if not (os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")):
            os.environ["KAGGLE_KEY"] = c["key"]
            if str(c["key"]).startswith("KGAT_"):
                os.environ["KAGGLE_API_TOKEN"] = c["key"]
        try:
            kp = os.path.expanduser("~/.kaggle/kaggle.json")
            if not os.path.exists(kp):
                os.makedirs(os.path.dirname(kp), exist_ok=True)
                with open(kp, "w") as f:
                    json.dump({"username": c["username"], "key": c["key"]}, f)
                os.chmod(kp, 0o600)
        except Exception:
            pass
    print("SHAM_SECRETS_FALLBACK_V3: بيانات كاكل", "جاهزة في البيئة" if c else "غير موجودة")
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe3", False):
        return

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _load=load_creds, **k):
        try:
            return _prev(self, label, *a, **k)
        except BaseException as e:
            if isinstance(e, KeyboardInterrupt):
                raise
            if label == "GITHUB_TOKEN":
                return ""
            if label in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                d = _load()
                if d:
                    return d["username"] if label == "KAGGLE_USERNAME" else d["key"]
            raise

    get_secret._sham_safe3 = True
    get_secret._sham_safe2 = True  # يمنع V2 من إعادة التغليف لو أعيد تشغيلها
    get_secret._sham_safe = True   # ويمنع V1 كذلك
    UserSecretsClient.get_secret = get_secret

try:
    _sham_secrets_fallback_v3()
except Exception as _sf3_exc:
    print("SHAM_SECRETS_FALLBACK_V3 غير مفعّل:", type(_sf3_exc).__name__)


In [ ]:
# SHAM_QUIET_START_V4 (2026-10-15)
# المالك لاحظ أن اللون الأصفر/البرتقالي للدفتر يظهر حين يكتب الدفتر أخطاء/تحذيرات
# (stderr) ثم يفشل لاحقاً. هذه الخلية تمنع الأخطاء المتوقعة من مصدرها، ولا تغيّر
# أي إعداد للنموذج: (1) تتذكّر السر الذي رفضه كاكل (HTTP 400) فلا تعيد طلبه وتطبع
# الخطأ كل مرة، وتعطي KAGGLE_* من البيئة مباشرة دون طلب الخادم، (2) تضبط
# __main__.__spec__ = None حتى يعمل multiprocessing(spawn) بلا AttributeError،
# (3) تخفّض تحذيرات HuggingFace المتوقعة (طلبات بلا توكن، أشرطة التقدم).
def _sham_quiet_start_v4():
    import os, sys, logging, warnings
    m = sys.modules.get("__main__")
    if m is not None and getattr(m, "__spec__", "missing") == "missing":
        try:
            m.__spec__ = None
        except Exception:
            pass
    os.environ.setdefault("HF_HUB_VERBOSITY", "error")
    os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
    os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    warnings.filterwarnings("ignore", message=".*unauthenticated requests to the HF Hub.*")
    for n in ("huggingface_hub", "datasets", "fsspec", "urllib3"):
        logging.getLogger(n).setLevel(logging.ERROR)
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe4", False):
        return
    failed = set()

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _failed=failed, **k):
        env = {"KAGGLE_USERNAME": os.environ.get("KAGGLE_USERNAME"),
               "KAGGLE_KEY": os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")}
        if label in env and env[label]:
            return env[label]            # جاهزة من V3 / sham-runtime-creds: لا طلب ولا خطأ 400
        if label in _failed and label == "GITHUB_TOKEN":
            return ""
        try:
            return _prev(self, label, *a, **k)
        except KeyboardInterrupt:
            raise
        except BaseException:
            _failed.add(label)
            if label == "GITHUB_TOKEN":
                return ""
            raise

    for attr in ("_sham_safe4", "_sham_safe3", "_sham_safe2", "_sham_safe"):
        setattr(get_secret, attr, True)
    UserSecretsClient.get_secret = get_secret

try:
    _sham_quiet_start_v4()
    print("SHAM_QUIET_START_V4: مفعّل")
except Exception as _q4_exc:
    print("SHAM_QUIET_START_V4 غير مفعّل:", type(_q4_exc).__name__)


# 🗣️ شام — دفتر التدريب العربي الإضافي (SFT) — نسخة منفصلة

دفتر **جديد ومنفصل** (أُضيف 2026-10-15 بموافقة المالك). يأخذ **نسخة** من آخر `final_chat.pt` في `sham-chat-checkpoint` ويدرّبها على مجموعات حوار/تعليمات عربية عامة كبيرة، كي يتعلّم شام الرد العربي المترابط.

- لا يلمس دفاتر المهندس الثلاثة ولا مجموعات نقاطها ولا إعدادات النموذج: يقرأ النقطة فقط، وينشر إلى مجموعة **جديدة** `sham-chat-arabic-sft-checkpoint`.
- نفس كود المهندس حرفياً: `load_checkpoint` / `save_checkpoint` / `ShamTextTokenizer` / `build_chat_example` / `batch_examples` / `train` / `sham_eval` / `publish_dataset`، ونفس اسم الملف `final_chat.pt` ونفس خلايا V3/V4.
- كل تشغيل يستأنف من نقطته هو (إن وُجدت)، وإلا يبدأ من نسخة نقطة المحادثة.
- 2% من الأمثلة (باختيار ثابت بالبصمة) **لا يُدرَّب عليها أبداً**: خسارتها + عيّنات توليد تُسجَّل عند كل حفظ — فهم أم حفظ.
- GPU أو CPU تلقائياً.


### 1) سحب كود شام من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


In [ ]:
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


### 2) تحميل نسخة نقطة المحادثة + التحقق من المفاتيح والأشكال

In [ ]:
import time
SESSION_START = time.time()
import json, shutil, torch
from pathlib import Path
from checkpoint import load_checkpoint
from text_tokenizer import ShamTextTokenizer
from sham_inputs import fetch_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"الجهاز: {device}")

OUT_DS = "sham-chat-arabic-sft-checkpoint"          # مجموعة هذا الدفتر وحده — لا تشارك أي اسم مع نقاط المهندس
SFT_OWN = fetch_dataset(OUT_DS)
_own_ck = sorted(SFT_OWN.rglob("final_chat.pt")) if SFT_OWN else []
CHAT_SRC = fetch_dataset("sham-chat-checkpoint")      # قراءة فقط
_src_ck = sorted(CHAT_SRC.rglob("final_chat.pt")) if CHAT_SRC else []
assert _own_ck or _src_ck, "لا توجد final_chat.pt في sham-chat-checkpoint ولا في " + OUT_DS

if _own_ck:
    base_ckpt, BASE_DIR = _own_ck[0], _own_ck[0].parent
    print("استئناف من نقطة هذا الدفتر السابقة.")
else:
    base_ckpt, BASE_DIR = _src_ck[0], _src_ck[0].parent
    print("أول تشغيل: نسخة من final_chat.pt في sham-chat-checkpoint.")

# — التحقق قبل التدريب: نفس المفاتيح والأشكال ونفس الإعدادات، وإلا نتوقف ونبلّغ (لا إجبار)
_raw = torch.load(base_ckpt, map_location="cpu", weights_only=False)
model, start_step, _extra = load_checkpoint(base_ckpt, map_location=device)
_sd = model.state_dict()
_mism = [k for k in set(_raw["model_state_dict"]) | set(_sd)
         if k not in _sd or k not in _raw["model_state_dict"] or tuple(_sd[k].shape) != tuple(_raw["model_state_dict"][k].shape)]
if _own_ck and _src_ck:
    _ref = torch.load(_src_ck[0], map_location="cpu", weights_only=False)
    if _ref["config"] != _raw["config"]:
        _mism.append(f"config differs from sham-chat-checkpoint: { {k: (_raw['config'].get(k), _ref['config'].get(k)) for k in set(_raw['config']) | set(_ref['config']) if _raw['config'].get(k) != _ref['config'].get(k)} }")
    for k, v in _ref["model_state_dict"].items():
        if k not in _sd or tuple(_sd[k].shape) != tuple(v.shape):
            _mism.append(f"shape/key vs source: {k}")
    del _ref
assert not _mism, "عدم تطابق في النقطة — توقف دون تدريب:\n" + "\n".join(map(str, _mism[:40]))
print(f"✓ النقطة مطابقة: {len(_sd)} مفتاح، config={_raw['config']}")
del _raw

tokenizer = ShamTextTokenizer.load(str(sorted(BASE_DIR.rglob("sham_small_tokenizer.json"))[0]))
tokenizer.save("/kaggle/working/sham_small_tokenizer.json")
for _f in ("image_tokenizer.pt", "audio_tokenizer.pt"):        # تُنقل كما هي مع النقطة (نفس محتوى مجموعة المحادثة)
    _p = sorted(BASE_DIR.rglob(_f))
    if _p:
        shutil.copy2(_p[0], Path("/kaggle/working") / _f)
print(f"نقطة الانطلاق: {base_ckpt} — الخطوة {start_step:,}، vocab={tokenizer.vocab_size:,}")
progress = {}
if SFT_OWN:
    _pp = sorted(SFT_OWN.rglob("sft_progress.json"))
    if _pp:
        progress = json.loads(_pp[0].read_text(encoding="utf-8"))
print("التقدّم السابق:", progress)


### 3) البيانات العربية (بثّ من Hugging Face) + 2% محجوبة

In [ ]:
import hashlib, os, random
from sham_chat import build_chat_example, batch_examples, _conversation_pair, _conversation_turns, build_multiturn_example

MAX_PAIRS = int(os.environ.get("SHAM_SFT_PAIRS", "24000"))   # أمثلة جديدة في كل جلسة (موزّعة على المصادر)
CHAT_BATCH = 8
MAX_LEN = min(1024, model.cfg.max_seq_len)

def _msgs_pair(row):            # صيغة messages: [{"role","content"}]
    m = row.get("messages") or []
    turns = []
    for a, b in zip(m, m[1:]):
        if isinstance(a, dict) and isinstance(b, dict) and a.get("role") == "user" and b.get("role") == "assistant":
            turns.append(((a.get("content") or "").strip(), (b.get("content") or "").strip()))
    return turns

def _instr(row):                # صيغة alpaca: instruction + input → output
    q = (row.get("instruction") or "").strip()
    if (row.get("input") or "").strip():
        q = q + "\n" + row["input"].strip()
    return [(q, (row.get("output") or "").strip())]

def _io(row):
    return [((row.get("inputs") or "").strip(), (row.get("targets") or "").strip())]

def _conv(row):
    t = _conversation_turns(row)
    return t or ([_conversation_pair(row)] if _conversation_pair(row)[0] else [])

from datasets import load_dataset
# (مفتاح، محمّل بثّ، مستخرج). البثّ دائماً: لا تنزيل كامل لأي مجموعة.
SFT_PLAN = [
    ("cidar",          lambda: load_dataset("arbml/CIDAR", split="train", streaming=True), _instr),
    ("aya_ar",         lambda: load_dataset("CohereLabs/aya_dataset", split="train", streaming=True).filter(lambda r: "Arabic" in (r.get("language") or "")), _io),
    ("alpaca_gpt4_ar", lambda: load_dataset("FreedomIntelligence/alpaca-gpt4-arabic", split="train", streaming=True), _conv),
    ("evol_ar",        lambda: load_dataset("FreedomIntelligence/evol-instruct-arabic", split="train", streaming=True), _conv),
    ("openhermes_ar",  lambda: load_dataset("2A2I/Arabic-OpenHermes-2.5", split="train", streaming=True), _conv),
    ("najdi",          lambda: load_dataset("HeshamHaroon/saudi-dialect-conversations", split="train", streaming=True), _msgs_pair),
    ("bactrian_ar",    lambda: load_dataset("MBZUAI/Bactrian-X", "ar", split="train", streaming=True), _instr),
    ("aya_coll_ar",    lambda: load_dataset("CohereLabs/aya_collection_language_split", "standard_arabic", split="train", streaming=True), _io),
    ("hala_sft",       lambda: load_dataset("hammh0a/Hala-4.6M-SFT", split="train", streaming=True), _conv),
]
_only = {k.strip() for k in os.environ.get("SHAM_SFT_SOURCES", "").split(",") if k.strip()}
if _only:
    SFT_PLAN = [p for p in SFT_PLAN if p[0] in _only]

def is_holdout(q):              # 2% ثابتة عبر كل الجلسات: نفس السؤال دائماً في التحقق، لا يُدرَّب عليه أبداً
    return int(hashlib.sha1(" ".join(q.split()).encode("utf-8")).hexdigest()[:8], 16) % 50 == 0

train_examples, holdout_pairs, seen = [], [], set()
share = max(1, MAX_PAIRS // max(1, len(SFT_PLAN)))
for key, loader, extract in SFT_PLAN:
    start = int(progress.get(key, 0)); n_rows = 0; got = 0
    try:
        for row in loader().skip(start):
            n_rows += 1
            turns = [(q, a) for q, a in extract(row) if q and a]
            if turns:
                q0 = " ".join(turns[0][0].split())
                if q0 not in seen:
                    seen.add(q0)
                    if is_holdout(q0):
                        holdout_pairs.append(turns[0])
                    elif len(turns) > 1:
                        train_examples.append(build_multiturn_example([(tokenizer.encode(q), tokenizer.encode(a)) for q, a in turns], max_len=MAX_LEN))
                        got += 1
                    else:
                        train_examples.append(build_chat_example(tokenizer.encode(turns[0][0]), tokenizer.encode(turns[0][1]), max_len=MAX_LEN))
                        got += 1
            if got >= share:
                break
        if got == 0 and start > 0:                         # انتهت: دورة جديدة على المصدر
            progress[f"{key}_epoch"] = int(progress.get(f"{key}_epoch", 0)) + 1
            n_rows = -start
    except Exception as exc:                                # مصدر معطّل لا يوقف الجلسة
        print(f"⚠ تعذّر {key}: {type(exc).__name__}: {str(exc)[:160]}")
    progress[key] = start + n_rows
    print(f"{key}: صفوف {start:,}..{start + max(n_rows,0):,} — {got:,} مثال تدريب")
holdout_examples = [build_chat_example(tokenizer.encode(q), tokenizer.encode(a), max_len=MAX_LEN) for q, a in holdout_pairs]
# أسئلة تحقق ثابتة دائمة: محفوظة مع النقطة كي تُقاس كل جلسة على نفس الأسئلة غير المرئية
_fixed = progress.get("fixed_holdout") or [list(p) for p in holdout_pairs[:64]]
progress["fixed_holdout"] = _fixed
fixed_examples = [build_chat_example(tokenizer.encode(q), tokenizer.encode(a), max_len=MAX_LEN) for q, a in _fixed]
print(f"تدريب: {len(train_examples):,} — تحقق محجوب هذه الجلسة: {len(holdout_examples):,} — تحقق ثابت: {len(fixed_examples):,}")
all_batches = batch_examples(train_examples, CHAT_BATCH, seed=start_step)
# 2026-10-17 (review): the SFT saw chat only, so every session pushed the general text loss up and the merge gate refused the result (it measures the
# general mixture). 40% of the batches are now rehearsal windows of that same mixture (sham_chat.text_replay_batches → sham_token_shards).
try:
    from sham_chat import text_replay_batches, interleave
    _n_rep = max(1, int(len(all_batches) * 0.4))
    _rep = text_replay_batches(tokenizer, MAX_LEN, CHAT_BATCH, _n_rep, "/kaggle/working/replay", seed=start_step)
    if _rep:
        all_batches = interleave([all_batches, _rep], [1.0, len(_rep) / max(1, len(all_batches))], seed=start_step)
        print(f"دفعات التدريب: {len(all_batches):,} (منها {len(_rep):,} تذكّر نص عام)")
except Exception as _exc:
    print(f"⚠ تذكّر النص العام لم يعمل ({type(_exc).__name__}: {str(_exc)[:140]}) — التدريب على المحادثة وحدها")

### 4) القياس قبل الجلسة

In [ ]:
from sham_eval import evaluate, fixed_eval_pairs, format_eval
from sham_merge import batches_loss
from sham_decoding import chat_prompt_ids
from generate import generate_tokens
from model import SpecialTokens

EVAL_PAIRS = fixed_eval_pairs()
SAMPLE_QS = [q for q, _ in _fixed[:4]] + ["ما هي عاصمة سوريا؟", "اكتب جملة عن أهمية القراءة.", "كيف حالك اليوم؟"]

def sft_measure(tag, step):
    model.train(False)
    r = {"step": step,
         "holdout_fixed_loss": round(batches_loss(model, batch_examples(fixed_examples, CHAT_BATCH), device), 4) if fixed_examples else None,
         "holdout_session_loss": round(batches_loss(model, batch_examples(holdout_examples[:256], CHAT_BATCH), device), 4) if holdout_examples else None,
         "samples": []}
    for q in SAMPLE_QS:
        ids = torch.tensor([chat_prompt_ids(tokenizer.encode(q)[-300:])], dtype=torch.long, device=device)
        out = generate_tokens(model, ids, max_new_tokens=60, temperature=0.7, top_k=40, top_p=0.9, eos_id=SpecialTokens.EOS,
                              allowed_ranges=[(0, tokenizer.vocab_size)] * 60)
        r["samples"].append({"q": q, "a": tokenizer.decode([i for i in out[0, ids.shape[1]:].tolist() if i < tokenizer.vocab_size and i != SpecialTokens.EOS])})
    print(f"📏 {tag} (خطوة {step:,}): خسارة التحقق الثابت={r['holdout_fixed_loss']} — المحجوب={r['holdout_session_loss']}")
    for s in r["samples"]:
        print(f"  س: {s['q']}\n  ج: {s['a']}")
    model.train()
    return r

model.to(device)
eval_before = evaluate(model, tokenizer, device, chat=True, pairs=EVAL_PAIRS)
print(format_eval("قبل الجلسة (SFT العربي)", start_step, eval_before))
sft_before = sft_measure("قبل التدريب", start_step)


### 5) التدريب (GPU أو CPU)

In [ ]:
from train import TrainConfig, train, build_optimizer

GRAD_ACCUM = 2
SESSION_HOURS = 5.0 if device.startswith("cuda") else 11.0   # نفس حدّ دفتر المحادثة
RESERVE_SECONDS = 40 * 60                                     # للقياس والنشر
optimizer = build_optimizer(model, lr=5e-5, weight_decay=0.1)

t0 = time.time()
model.train()
for _b in all_batches[:5]:
    ids, labels = _b
    _, loss = model(ids.to(device), labels=labels.to(device))
    loss.backward()
optimizer.zero_grad()
per_batch = max((time.time() - t0) / min(5, len(all_batches)), 1e-3)
train_seconds = max(600, SESSION_HOURS * 3600 - (time.time() - SESSION_START) - RESERVE_SECONDS)
session_batches = all_batches[: int(train_seconds * 0.9 / per_batch)]
TOTAL_STEPS = max(len(session_batches) // GRAD_ACCUM, 1)
print(f"~{per_batch:.2f} ث/دفعة على {device} — {len(session_batches):,} دفعة = {TOTAL_STEPS:,} خطوة تحديث.")

train_cfg = TrainConfig(
    seq_len=MAX_LEN, batch_size=CHAT_BATCH, grad_accum_steps=GRAD_ACCUM, lr=5e-5,
    warmup_steps=max(10, TOTAL_STEPS // 50), total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints", checkpoint_every=200, log_every=20,
    max_wall_clock_seconds=train_seconds,
)
loss_history = train(model, session_batches, train_cfg, device=device, start_step=start_step, resume_optimizer=optimizer)
final_step = start_step + len(loss_history)
print(f"انتهى: {len(loss_history):,} خطوة — الخطوة النهائية {final_step:,}")


### 6) القياس بعد الجلسة

In [ ]:
eval_after = evaluate(model, tokenizer, device, chat=True, pairs=EVAL_PAIRS)
print(format_eval("SFT العربي", final_step, eval_after, before=eval_before))
sft_after = sft_measure("بعد التدريب", final_step)


### 7) الحفظ والنشر إلى `sham-chat-arabic-sft-checkpoint` (خاصة)

In [ ]:
from checkpoint import save_checkpoint
from sham_inputs import publish_dataset

save_checkpoint("/kaggle/working/checkpoints/final_chat.pt", model, final_step)
upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
shutil.copy2("/kaggle/working/checkpoints/final_chat.pt", upload_dir / "final_chat.pt")
for _f in ("sham_small_tokenizer.json", "image_tokenizer.pt", "audio_tokenizer.pt"):
    if (Path("/kaggle/working") / _f).exists():
        shutil.copy2(Path("/kaggle/working") / _f, upload_dir / _f)
(upload_dir / "sft_progress.json").write_text(json.dumps(progress, ensure_ascii=False), encoding="utf-8")
_hist = []
if SFT_OWN:
    _hp = sorted(SFT_OWN.rglob("sft_eval_history.json"))
    if _hp:
        _hist = json.loads(_hp[0].read_text(encoding="utf-8"))
_hist.append({"step": final_step, "base": str(base_ckpt), "device": device,
              "before": {k: v for k, v in eval_before.items() if k != "samples"},
              "after": {k: v for k, v in eval_after.items() if k != "samples"},
              "sft_before": sft_before, "sft_after": sft_after, "train_loss_last": loss_history[-20:] if loss_history else []})
(upload_dir / "sft_eval_history.json").write_text(json.dumps(_hist, ensure_ascii=False, default=float), encoding="utf-8")
publish_dataset(upload_dir, OUT_DS, f"arabic sft at step {final_step:,}")
